## Restrictions Approach

### Calculo de Dependencia&Independencia de nodos en una red bayesiana

- Mutual Information 
- Conditional Mutual Information
- Chi_square test

In [1]:
import numpy as np

def mutual_info(X, Y):
    """
    Calculate the mutual information between two discrete random variables.
    
    Returns:
        float: The mutual information value.
    """
    # Ensure X and Y are numpy arrays
    X = np.asarray(X)   
    Y = np.asarray(Y)
    # Check if X and Y are binary (0 or 1)
    if not (np.all(np.isin(X, [0, 1])) and np.all(np.isin(Y, [0, 1]))):
        raise ValueError("X and Y must be binary (0 or 1).")
    # Calculate joint probability distribution
    joint_prob = np.zeros((2, 2))
    for x, y in zip(X, Y):
        joint_prob[x, y] += 1
    joint_prob /= len(X)
    
    # Calculate marginal probability distributions
    prob_X = joint_prob.sum(axis=1)
    prob_Y = joint_prob.sum(axis=0)
    
    # Calculate mutual information
    MI = 0.0
    for i in range(2):
        for j in range(2):
            if joint_prob[i, j] > 0:
                MI += joint_prob[i, j] * np.log2(joint_prob[i, j] / (prob_X[i] * prob_Y[j]))
    
    return MI

#----------------------------------------------------------------------#
#----------------------------------------------------------------------#

def conditional_mutual_info(X, Y, Z):
    """
    Calculate the conditional mutual information I(X;Y|Z).
    
    Returns:
        float: The conditional mutual information value.
    """
    # Ensure X, Y, and Z are numpy arrays
    X = np.asarray(X)
    Y = np.asarray(Y)
    Z = np.asarray(Z)
    
    # Check if X, Y, and Z are binary (0 or 1)
    if not (np.all(np.isin(X, [0, 1])) and np.all(np.isin(Y, [0, 1])) and np.all(np.isin(Z, [0, 1]))):
        raise ValueError("X, Y, and Z must be binary (0 or 1).")
    
    # Calculate joint probability distribution
    joint_prob = np.zeros((2, 2, 2))
    for x, y, z in zip(X, Y, Z):
        joint_prob[x, y, z] += 1
    joint_prob /= len(X)
    
    # Calculate marginal probability distributions
    prob_X_Z = joint_prob.sum(axis=2)# P(X|Z)
    prob_Y_Z = joint_prob.sum(axis=0)# P(Y|Z)
    
    # Calculate conditional mutual information
    CMI = 0.0
    for i in range(2):
        for j in range(2):
            for k in range(2):
                if joint_prob[i, j, k] > 0:
                    # Evitar división por cero
                    if np.all(prob_X_Z[i]) > 0 and np.all(prob_Y_Z[j]) > 0:
                        CMI += joint_prob[i, j, k] * np.log2(joint_prob[i, j, k] / (prob_X_Z[i] * prob_Y_Z[j]))
    
    return CMI, prob_X_Z, prob_Y_Z

#----------------------------------------------------------------------#
#----------------------------------------------------------------------#

def chi_square_test(X, Y):
    """
    Perform the Chi-Square test for independence between two categorical variables.
    
    Returns:
        float: The Chi-Square statistic.
        float: The p-value.
    """
    from scipy.stats import chi2_contingency
    
    # Create a contingency table
    contingency_table = np.zeros((2, 2))
    for x, y in zip(X, Y):
        contingency_table[x, y] += 1
    
    # Perform the Chi-Square test
    chi2_stat, p_value, _, _ = chi2_contingency(contingency_table)
    
    return chi2_stat, p_value

In [2]:
res=(4/10)*np.log2((4/10)/(25/100))+(1/10)*np.log2((1/10)/(25/100))+(1/10)*np.log2((1/10)/(25/100))+(4/10)*np.log2((4/10)/(25/100))
print(res)

0.27807190511263774


In [3]:
X1 = np.array([1, 1, 0, 1, 0, 0, 1, 0, 1, 0])
X2 = np.array([0, 1, 0, 1, 0, 1, 1, 0, 1, 0])
X3 = np.array([0, 1, 1, 1, 0, 1, 1, 0, 1, 0])
MIx12 = mutual_info(X2, X3)
CMI, prob_X_Z, prob_Y_Z = conditional_mutual_info(X1, X2, X3)

print(f'Informacion Mutua entre X1 y X2 -> {MIx12}')
print(f'Informacion Condicional Mutua entre X1 y X2 dado X3 -> {CMI}')

print(f'Probabilidad marginal de X1 dado X3 -> \n {prob_X_Z}')
print(f'Probabilidad marginal de X2 dado X3 -> \n {prob_Y_Z}')

Informacion Mutua entre X1 y X2 -> 0.6099865470109875
Informacion Condicional Mutua entre X1 y X2 dado X3 -> [0.3364528 1.9364528]
Probabilidad marginal de X1 dado X3 -> 
 [[0.4 0.1]
 [0.1 0.4]]
Probabilidad marginal de X2 dado X3 -> 
 [[0.4 0.1]
 [0.  0.5]]


In [4]:
chi_statX12,pX12 = chi_square_test(X1, X2)
chi_statX13,pX13 =chi_square_test(X1, X3)

In [5]:
import numpy as np
from scipy.stats import chi2_contingency
# Paso 1: Crear la tabla de contingencia
# Supongamos que tenemos dos variables categóricas
# Ejemplo: Variable A (tipo de tratamiento) y Variable B (resultado)
# La tabla podría ser algo así:
#                |  Exito  | Fracaso |
# ---------------|---------|---------|
# Tratamiento 1  |    30   |   10    |
# Tratamiento 2  |    20   |   20    |
#observados = np.array([[30, 10],[20, 20]])

#      | Abscent | Present |
# -----|---------|---------|
# X 1  |    5    |    5    |
# X 2  |    5    |    5    |
observados = np.array([[43, 23],
                       [33, 33]])
# Paso 2: Realizar la prueba chi cuadrada
chi2, p, dof, expected = chi2_contingency(observados)
# Paso 3: Imprimir los resultados
print(f'Estadístico chi cuadrada: {chi2}')
print(f'Valor p: {p}')
print(f'Degrees of freedom: {dof}')
print(f'Frecuencias esperadas: {expected}')
# Paso 4: Interpretar los resultados
alpha = 0.05  # Nivel de significancia
if p < alpha:
    print("Se rechaza la hipótesis nula: las variables son dependientes.")
else:
    print("No se rechaza la hipótesis nula: las variables son independientes.")

Estadístico chi cuadrada: 2.512218045112782
Valor p: 0.11296683275014073
Degrees of freedom: 1
Frecuencias esperadas: [[38. 28.]
 [38. 28.]]
No se rechaza la hipótesis nula: las variables son independientes.


Interpretación de los resultados:

- Si el valor p es menor que el nivel de significancia (0.05), puedes rechazar la hipótesis nula y concluir que hay una relación significativa entre las dos variables (son dependientes).

- Si el valor p es mayor que 0.05, no puedes rechazar la hipótesis nula, lo que sugiere que no hay evidencia suficiente para afirmar que las variables son dependientes (son independientes).

Recuerda que es importante verificar que los supuestos de la prueba chi cuadrada se cumplan, como tener un tamaño de muestra adecuado y que las frecuencias esperadas sean suficientemente grandes.

In [6]:
X1 = np.array([1, 1, 0, 1, 0, 0, 1, 0, 1, 0])
X2 = np.array([0, 1, 0, 1, 0, 1, 1, 0, 1, 0])
joint = np.zeros((2, 2))
for x, y in zip(X1, X2):
    joint[x, y] += 1
    print(f'{joint}\n')

[[0. 0.]
 [1. 0.]]

[[0. 0.]
 [1. 1.]]

[[1. 0.]
 [1. 1.]]

[[1. 0.]
 [1. 2.]]

[[2. 0.]
 [1. 2.]]

[[2. 1.]
 [1. 2.]]

[[2. 1.]
 [1. 3.]]

[[3. 1.]
 [1. 3.]]

[[3. 1.]
 [1. 4.]]

[[4. 1.]
 [1. 4.]]

